# Advanced Pitch Statistics from Statcast

This notebook explores the advanced pitch-level data used to build pitcher and batter profiles.

**Pitcher Arsenal Stats:**
- Velocity, spin, movement by pitch type
- Pitch usage percentages
- Whiff%, CSW%, zone%, chase induced

**Batter Profile Stats:**
- Performance vs pitch types (fastball, breaking, offspeed)
- Performance vs velocity buckets (90-93, 94-96, 97+)
- Performance vs movement profiles (high rise, heavy sweep, heavy drop)
- Contact quality (exit velo, xwOBA)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np

from mlb_data import (
    get_statcast_pitches,
    get_pitcher_arsenal,
    get_pitcher_pitch_type_stats,
    get_batter_pitch_stats,
    get_batter_pitch_type_stats,
    get_plate_appearances,
)

# Import config
from src.config import (
    DATA_START,
    DATA_END,
    DATA_DIR,
)

pd.set_option('display.max_columns', 50)

print(f"Date range: {DATA_START} to {DATA_END}")

## 1. Raw Statcast Data

First, let's fetch the raw pitch-level data and explore what's available.

In [ ]:
from pathlib import Path

# Load pitches from notebook 01 output
pitches_path = Path(f"../{DATA_DIR}/raw/pitches.parquet")

if pitches_path.exists():
    pitches = pd.read_parquet(pitches_path)
    print(f"Loaded pitches from {pitches_path}")
else:
    print(f"Pitches file not found at {pitches_path}")
    print("Run notebook 01_compile_data first to fetch the data.")
    raise FileNotFoundError(f"Run notebook 01 first: {pitches_path}")

print(f"Shape: {pitches.shape}")
print(f"Date range: {pitches['game_date'].min()} to {pitches['game_date'].max()}")
pitches.head()

In [3]:
# Key pitch characteristic columns
pitch_cols = [
    'pitch_type', 'release_speed', 'release_spin_rate',
    'pfx_x', 'pfx_z', 'release_extension',
    'plate_x', 'plate_z', 'zone',
    'description', 'events', 'type',
]

pitches[pitch_cols].head(20)

,pitch_type,release_speed,release_spin_rate,pfx_x,pfx_z,release_extension,plate_x,plate_z,zone,description,events,type
0,FF,95.5,2470,-0.82,1.23,6.8,-0.35354,2.709695,4,hit_into_play,field_out,X
1,FF,96.6,2375,-0.81,1.47,7.0,0.873445,3.056491,12,called_strike,None,S
2,SL,88.1,2585,0.29,-0.11,6.8,0.603729,1.972011,9,hit_into_play,single,X
3,FF,97.2,2456,-0.88,1.28,6.9,0.632673,3.917028,3,ball,None,B
4,FF,96.9,2465,-0.9,1.29,7.0,0.693248,2.881546,6,foul,None,S
5,SL,88.8,2687,0.53,-0.17,7.0,0.741418,1.460667,14,ball,None,B
6,SL,88.3,2463,0.31,-0.15,6.7,0.011636,2.628903,5,foul,None,S
7,SI,95.8,2379,-1.41,0.96,7.0,-0.165987,0.726876,13,hit_into_play,field_out,X
8,SL,89.2,2663,0.35,0.03,6.8,0.444063,0.45227,14,ball,None,B
9,SL,86.2,2556,0.27,0.12,6.8,-0.500545,2.94893,1,foul,None,S


In [4]:
# Pitch type distribution
print("Pitch type distribution:")
pitch_counts = pitches['pitch_type'].value_counts()
pitch_pcts = pitches['pitch_type'].value_counts(normalize=True) * 100

pd.DataFrame({
    'count': pitch_counts,
    'pct': pitch_pcts.round(1)
}).head(15)

Pitch type distribution:


,count,pct
pitch_type,,
FF,5906,30.9
SI,3083,16.1
SL,2897,15.2
CH,1806,9.4
FC,1565,8.2
ST,1402,7.3
CU,1331,7.0
FS,677,3.5
KC,343,1.8


In [5]:
# Pitch outcome descriptions
print("Pitch descriptions (outcomes):")
pitches['description'].value_counts()

Pitch descriptions (outcomes):


description
ball                       6490
foul                       3321
hit_into_play              3309
called_strike              3266
swinging_strike            1942
blocked_ball                386
foul_tip                    197
swinging_strike_blocked     128
hit_by_pitch                 48
automatic_ball               45
foul_bunt                    28
missed_bunt                   3
automatic_strike              2
Name: count, dtype: int64

## 2. Pitcher Arsenal Profiles

Aggregate pitch-level data into pitcher profiles with:
- Fastball characteristics (velocity, spin, movement)
- Breaking ball characteristics
- Offspeed characteristics
- Overall effectiveness metrics (whiff%, CSW%, etc.)

In [ ]:
# Get pitcher arsenal stats (uses cached pitches)
pitcher_arsenal = get_pitcher_arsenal(
    DATA_START, DATA_END,
    min_pitches=50,  # Higher threshold for multi-season data
    pitches_df=pitches
)

print(f"Pitchers with arsenal data: {len(pitcher_arsenal)}")
pitcher_arsenal.head(10)

In [7]:
# All available arsenal columns
print("Pitcher arsenal columns:")
print(pitcher_arsenal.columns.tolist())

Pitcher arsenal columns:
['pitcher_id', 'pitcher_name', 'p_throws', 'total_pitches', 'primary_pitch', 'ff_usage_pct', 'ff_velo_avg', 'ff_spin_avg', 'ff_vmov_avg', 'ff_hmov_avg', 'ff_whiff_pct', 'si_usage_pct', 'fc_usage_pct', 'sl_usage_pct', 'cu_usage_pct', 'cu_velo_avg', 'cu_spin_avg', 'cu_vmov_avg', 'cu_hmov_avg', 'cu_whiff_pct', 'ch_usage_pct', 'sv_usage_pct', 'fs_usage_pct', 'kc_usage_pct', 'st_usage_pct', 'kn_usage_pct', 'whiff_pct', 'swstr_pct', 'csw_pct', 'zone_pct', 'chase_pct_induced', 'contact_pct', 'f_strike_pct', 'fc_velo_avg', 'fc_spin_avg', 'fc_vmov_avg', 'fc_hmov_avg', 'fc_whiff_pct', 'sl_velo_avg', 'sl_spin_avg', 'sl_vmov_avg', 'sl_hmov_avg', 'sl_whiff_pct', 'si_velo_avg', 'si_spin_avg', 'si_vmov_avg', 'si_hmov_avg', 'si_whiff_pct', 'fs_velo_avg', 'fs_spin_avg', 'fs_vmov_avg', 'fs_hmov_avg', 'fs_whiff_pct', 'st_velo_avg', 'st_spin_avg', 'st_vmov_avg', 'st_hmov_avg', 'st_whiff_pct', 'ch_velo_avg', 'ch_spin_avg', 'ch_vmov_avg', 'ch_hmov_avg', 'ch_whiff_pct', 'kc_velo_avg'

In [10]:
# Summary statistics for key arsenal metrics
arsenal_stats = [
    'fb_velo_avg', 'fb_velo_max', 'fb_spin_avg', 'fb_vmov_avg',
    'brk_velo_avg', 'brk_hmov_avg', 'brk_vmov_avg',
    'off_velo_avg', 'off_velo_diff',
    'fb_usage_pct', 'brk_usage_pct', 'off_usage_pct',
    'whiff_pct', 'csw_pct', 'zone_pct', 'chase_pct_induced',
]

available_stats = [c for c in arsenal_stats if c in pitcher_arsenal.columns]
pitcher_arsenal[available_stats].describe().round(2)

,whiff_pct,csw_pct,zone_pct,chase_pct_induced
count,380.00,380.00,380.00,380.00
mean,0.24,0.28,0.50,0.28
std,0.12,0.08,0.09,0.14
min,0.00,0.00,0.28,0.00
25%,0.15,0.23,0.45,0.20
50%,0.22,0.28,0.50,0.27
75%,0.31,0.32,0.55,0.35
max,0.62,0.57,0.80,1.00


## 3. Pitcher Stats by Pitch Type

Detailed breakdown for each pitch type a pitcher throws.

In [ ]:
# Get per-pitch-type stats for pitchers
pitcher_pitch_types = get_pitcher_pitch_type_stats(
    DATA_START, DATA_END,
    min_pitches=20,  # Higher threshold for multi-season data
    pitches_df=pitches
)

print(f"Pitcher-pitch type combinations: {len(pitcher_pitch_types)}")
pitcher_pitch_types.head(15)

In [13]:
# Best sliders by whiff rate
sliders = pitcher_pitch_types[pitcher_pitch_types['pitch_type'] == 'SL']
print("Top 10 sliders by whiff rate:")
sliders.nlargest(10, 'whiff_pct')[[
    'pitcher_name', 'pitch_count', 'usage_pct',
    'velo_avg', 'hmov_avg', 'vmov_avg', 'whiff_pct'
]]

Top 10 sliders by whiff rate:


,pitcher_name,pitch_count,usage_pct,velo_avg,hmov_avg,vmov_avg,whiff_pct
568,"Reid-Foley, Sean",6,0.166667,86.350000,0.473333,0.030000,1.000000
641,"McKenzie, Triston",5,0.055556,85.840000,0.390000,0.876000,1.000000
792,"Miller, Erik",10,0.192308,81.030000,-1.069000,-0.130000,1.000000
921,"Sears, JP",8,0.084211,78.737500,-0.690000,0.651250,1.000000
865,"Morejon, Adrian",9,0.450000,86.533333,-0.573333,-0.423333,0.833333
53,"Pressly, Ryan",7,0.179487,89.000000,0.478571,0.361429,0.800000
85,"Hudson, Daniel",11,0.407407,88.381818,-0.019091,0.441818,0.800000
552,"Hoffman, Jeff",14,0.411765,87.442857,0.387143,-0.170000,0.800000
758,"Doval, Camilo",8,0.571429,88.487500,0.248750,-0.072500,0.800000
911,"Walker, Ryan",15,0.535714,83.040000,1.500667,0.110667,0.714286


In [14]:
# Best changeups by whiff rate
changeups = pitcher_pitch_types[pitcher_pitch_types['pitch_type'] == 'CH']
print("Top 10 changeups by whiff rate:")
changeups.nlargest(10, 'whiff_pct')[[
    'pitcher_name', 'pitch_count', 'usage_pct',
    'velo_avg', 'vmov_avg', 'whiff_pct'
]]

Top 10 changeups by whiff rate:


,pitcher_name,pitch_count,usage_pct,velo_avg,vmov_avg,whiff_pct
259,"López, Jorge",6,0.111111,87.366667,0.215000,1.000000
263,"Musgrove, Joe",5,0.057471,87.680000,0.174000,1.000000
799,"Burnes, Corbin",8,0.084211,89.387500,0.420000,1.000000
974,"Vodnik, Victor",5,0.121951,89.920000,-0.046000,1.000000
1088,"Elder, Bryce",5,0.070423,84.780000,0.366000,1.000000
308,"Martinez, Nick",6,0.260870,80.100000,0.451667,0.800000
526,"Beeks, Jalen",9,0.409091,89.688889,0.883333,0.750000
276,"Ross, Joe",11,0.127907,90.218182,0.560909,0.666667
359,"Eflin, Zach",6,0.068182,86.500000,0.186667,0.666667
495,"Santana, Dennis",7,0.205882,88.814286,0.767143,0.666667


## 4. Batter Pitch Performance Profiles

How batters perform against different pitch characteristics:
- By pitch type group (fastball, breaking, offspeed)
- By velocity bucket (90-93, 94-96, 97+)
- By movement type (high rise, heavy sweep, heavy drop)
- By pitcher handedness (vs LHP, vs RHP)

In [ ]:
# Get batter pitch stats (uses cached pitches)
batter_profiles = get_batter_pitch_stats(
    DATA_START, DATA_END,
    min_pitches=50,  # Higher threshold for multi-season data
    pitches_df=pitches
)

print(f"Batters with pitch profiles: {len(batter_profiles)}")
batter_profiles.head(10)

In [19]:
# All available batter profile columns
print("Batter profile columns:")
print(batter_profiles.columns.tolist())

Batter profile columns:
['batter_id', 'batter_name', 'stand', 'total_pitches', 'swing_pct', 'whiff_pct', 'contact_pct', 'csw_pct', 'zone_swing_pct', 'chase_pct', 'vs_ff_whiff_pct', 'vs_ff_contact_pct', 'vs_RHP_pitches', 'vs_RHP_whiff_pct', 'vs_RHP_xwoba', 'vs_si_whiff_pct', 'vs_si_contact_pct', 'vs_si_xwoba', 'vs_ff_xwoba', 'vs_sl_whiff_pct', 'vs_sl_contact_pct', 'vs_ch_whiff_pct', 'vs_ch_contact_pct', 'vs_ch_xwoba', 'vs_LHP_pitches', 'vs_LHP_whiff_pct', 'vs_LHP_xwoba', 'velo_90_93_pitches', 'velo_90_93_whiff_pct', 'velo_90_93_swing_pct', 'vs_st_whiff_pct', 'vs_st_contact_pct', 'velo_94_96_pitches', 'velo_94_96_whiff_pct', 'velo_94_96_swing_pct', 'avg_exit_velo', 'max_exit_velo', 'avg_launch_angle', 'xwoba', 'xba', 'barrel_pct', 'vs_sl_xwoba']


In [20]:
# Summary of key batter metrics
batter_stats = [
    'whiff_pct', 'contact_pct', 'chase_pct', 'zone_swing_pct',
    'fastball_whiff_pct', 'breaking_whiff_pct', 'offspeed_whiff_pct',
    'velo_97_plus_whiff_pct', 'high_rise_whiff_pct', 'heavy_sweep_whiff_pct',
    'avg_exit_velo', 'xwoba',
]

available_stats = [c for c in batter_stats if c in batter_profiles.columns]
batter_profiles[available_stats].describe().round(3)

,whiff_pct,contact_pct,chase_pct,zone_swing_pct,avg_exit_velo,xwoba
count,388.000,388.000,388.000,388.000,2.000,2.000
mean,0.237,0.763,0.277,0.655,86.410,0.354
std,0.127,0.127,0.125,0.130,0.774,0.056
min,0.000,0.250,0.000,0.000,85.862,0.315
25%,0.155,0.700,0.200,0.576,86.136,0.335
50%,0.222,0.778,0.267,0.658,86.410,0.354
75%,0.300,0.845,0.346,0.742,86.683,0.374
max,0.750,1.000,1.000,1.000,86.957,0.394


In [21]:
# Best contact rates (lowest whiff)
print("Top 10 contact rates (lowest whiff%):")
batter_profiles.nsmallest(10, 'whiff_pct')[[
    'batter_id', 'stand', 'total_pitches',
    'whiff_pct', 'contact_pct', 'chase_pct', 'avg_exit_velo'
]]

Top 10 contact rates (lowest whiff%):


,batter_id,stand,total_pitches,whiff_pct,contact_pct,chase_pct,avg_exit_velo
37,571657,R,13,0.0,1.0,0.200000,NaN
96,621028,R,14,0.0,1.0,0.200000,NaN
100,621439,R,5,0.0,1.0,0.250000,NaN
126,641584,L,22,0.0,1.0,0.181818,NaN
139,642731,R,72,0.0,1.0,0.178571,NaN
147,644433,R,14,0.0,1.0,0.000000,NaN
195,663611,R,14,0.0,1.0,0.375000,NaN
263,668800,R,14,0.0,1.0,0.285714,NaN
274,669134,R,28,0.0,1.0,0.181818,NaN
305,672640,R,21,0.0,1.0,0.200000,NaN


In [22]:
# Batters who struggle vs 97+ velocity
if 'velo_97_plus_whiff_pct' in batter_profiles.columns:
    has_velo_data = batter_profiles['velo_97_plus_whiff_pct'].notna()
    print("Batters who struggle most vs 97+ mph:")
    batter_profiles[has_velo_data].nlargest(10, 'velo_97_plus_whiff_pct')[[
        'batter_id', 'stand', 'velo_97_plus_pitches',
        'velo_97_plus_whiff_pct', 'whiff_pct'
    ]]

In [23]:
# Batters who struggle vs breaking balls
if 'breaking_whiff_pct' in batter_profiles.columns:
    has_brk_data = batter_profiles['breaking_whiff_pct'].notna()
    print("Batters who struggle most vs breaking balls:")
    batter_profiles[has_brk_data].nlargest(10, 'breaking_whiff_pct')[[
        'batter_id', 'stand', 'breaking_pitches',
        'breaking_whiff_pct', 'whiff_pct'
    ]]

## 5. Batter Stats by Pitch Type

Detailed breakdown for each pitch type a batter faces.

In [ ]:
# Get per-pitch-type stats for batters
batter_pitch_types = get_batter_pitch_type_stats(
    DATA_START, DATA_END,
    min_pitches=50,  # Higher threshold for multi-season data
    pitches_df=pitches
)

print(f"Batter-pitch type combinations: {len(batter_pitch_types)}")
batter_pitch_types.head(15)

In [25]:
# Best fastball hitters (lowest whiff on FF)
ff_stats = batter_pitch_types[batter_pitch_types['pitch_type'] == 'FF']
if len(ff_stats) > 0 and 'xwoba' in ff_stats.columns:
    ff_qualified = ff_stats[ff_stats['pitch_count'] >= 100]
    print("Best fastball hitters by xwOBA:")
    ff_qualified.nlargest(10, 'xwoba')[[
        'batter_id', 'pitch_count', 'whiff_pct', 'contact_pct',
        'avg_exit_velo', 'xwoba'
    ]]

Best fastball hitters by xwOBA:


## 6. Plate Appearance Outcomes

Extract completed plate appearances with outcomes (K, BB, 1B, 2B, 3B, HR, OUT).

In [ ]:
# Get plate appearances with outcomes
plate_apps = get_plate_appearances(
    DATA_START, DATA_END,
    pitches_df=pitches
)

print(f"Total plate appearances: {len(plate_apps):,}")
plate_apps.head(15)

In [27]:
# Outcome distribution
print("PA outcome distribution:")
outcome_counts = plate_apps['outcome'].value_counts()
outcome_pcts = plate_apps['outcome'].value_counts(normalize=True) * 100

pd.DataFrame({
    'count': outcome_counts,
    'pct': outcome_pcts.round(1)
})

PA outcome distribution:


,count,pct
outcome,,
OUT,2301,47.0
K,1115,22.8
1B,663,13.5
BB,406,8.3
2B,199,4.1
HR,120,2.5
HBP,48,1.0
3B,25,0.5
OTHER,18,0.4


In [28]:
# Unique matchup counts
print(f"Unique pitchers: {plate_apps['pitcher_id'].nunique():,}")
print(f"Unique batters: {plate_apps['batter_id'].nunique():,}")

# Pitcher-batter combinations
matchups = plate_apps.groupby(['pitcher_id', 'batter_id']).size()
print(f"Unique pitcher-batter matchups: {len(matchups):,}")
print(f"\nPAs per matchup distribution:")
matchups.describe()

Unique pitchers: 382
Unique batters: 394
Unique pitcher-batter matchups: 2,976

PAs per matchup distribution:


count    2976.000000
mean        1.644825
std         0.804521
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         4.000000
dtype: float64

## 7. Save Data

Save the computed profiles for use in model training.

In [ ]:
from pathlib import Path

# Create output directory
output_dir = Path(f'../{DATA_DIR}/profiles')
output_dir.mkdir(parents=True, exist_ok=True)

# Save profiles
pitcher_arsenal.to_csv(output_dir / 'pitcher_arsenal.csv', index=False)
pitcher_pitch_types.to_csv(output_dir / 'pitcher_pitch_types.csv', index=False)
batter_profiles.to_csv(output_dir / 'batter_profiles.csv', index=False)
batter_pitch_types.to_csv(output_dir / 'batter_pitch_types.csv', index=False)
plate_apps.to_parquet(output_dir / 'plate_appearances.parquet', index=False)

print(f"Saved data to {output_dir.resolve()}:")
print(f"  - pitcher_arsenal.csv: {len(pitcher_arsenal):,} rows")
print(f"  - pitcher_pitch_types.csv: {len(pitcher_pitch_types):,} rows")
print(f"  - batter_profiles.csv: {len(batter_profiles):,} rows")
print(f"  - batter_pitch_types.csv: {len(batter_pitch_types):,} rows")
print(f"  - plate_appearances.parquet: {len(plate_apps):,} rows")